# List IG CFD instruments

IG has no single endpoint that returns every tradeable instrument. The only
documented way to discover epics is free-text market search -
`GET /markets?searchTerm=<term>` (`Version: 1`, see
[`doc/ig/epics.md`](doc/ig/epics.md)) - which matches against one search term
at a time.

This notebook runs that search for a list of `SEARCH_TERMS` (edit the list
below to cover the markets you care about - indices, FX pairs, commodities,
...), aggregates the results, de-duplicates by `epic`, and displays them as a
table. It is **not exhaustive** - it only finds instruments matching one of
`SEARCH_TERMS` for the logged-in account's environment (demo/live differ) -
it's a discovery tool, not a full instrument catalogue.

Standalone, like `05_yahoo_download_finance.ipynb`: it doesn't touch
`data/ig_market_data.db` and isn't part of the `01`-`04` download/import/view/
backtest pipeline. It does need IG credentials from `.env` (see the README) -
`IG_API_KEY` / `IG_USERNAME` / `IG_PASSWORD` / `IG_ACCOUNT_TYPE` - the same
ones `01_ig_download_prices.ipynb` uses; `IG_EPIC` / `IG_RESOLUTION` /
`IG_START` / `IG_END` are loaded as part of the same `Config` but unused
here.

## Flow

| # | section | what happens |
| --- | --- | --- |
| 1 | **Config** | load IG credentials from `.env` |
| 2 | **Parameters** | `SEARCH_TERMS` and `SAVE_CSV`, set directly in this notebook |
| 3 | **Search** | authenticate an `IGSession`, call `search_markets(term)` for each term, print a per-term count, de-duplicate by `epic` |
| 4 | **View** | the de-duplicated instruments as a pandas table, sorted by `instrumentName` |
| 5 | **Save** | when `SAVE_CSV` is true, write a timestamped `data/ig_instruments_<timestamp>.csv` |


## 1. Load config

Only the IG credentials are used here. `Config.from_env()`
(`trading_toolkit/ig_config.py`) still requires `IG_START` to be set in `.env`
(shared with the other notebooks) even though this notebook doesn't read
it.

In [4]:
from trading_toolkit.ig_config import Config

cfg = Config.from_env()  # credentials only - EPIC/RESOLUTION/START/END are unused here

print(f"Loaded config from {cfg.env_path.resolve()}")
print(f"user={cfg.username!r}  account_type={cfg.account_type!r}  api_key=***{cfg.api_key[-4:]}")


Loaded config from C:\repos\github.com\jupyter-notebooks\.env
user='justinjingsh-demo'  account_type='demo'  api_key=***9251


## 2. Parameters

`SEARCH_TERMS` - free-text terms passed to `GET /markets?searchTerm=`, one
call per term. IG matches loosely (instrument name, epic fragments, ...), so
broad terms like `"cash"` (undated stock indices) or a currency code pull in
many related markets at once. Edit this list for the markets you care
about.

In [5]:
SEARCH_TERMS = [
    "cash",         # undated ("Cash") stock indices, e.g. US Tech 100, Germany 40
    "USD",          # USD forex pairs
    "EUR",
    "GBP",
    "gold",
    "silver",
    "oil",
    "natural gas",
    "copper",
    "bitcoin",
]

SAVE_CSV = True


## 3. Search

`IGSession.search_markets(term)` (`trading_toolkit/ig_session.py`) wraps
`GET /markets?searchTerm=<term>` (`Version: 1`) and returns the raw
`markets` list. Results are merged into one dict keyed by `epic`, so a market
matched by more than one search term is only kept once.

In [6]:
from trading_toolkit.ig_session import IGSession

ig = IGSession(cfg.api_key, cfg.username, cfg.password, cfg.account_type)

instruments = {}  # epic -> market dict, de-duplicated across overlapping search terms
for term in SEARCH_TERMS:
    markets = ig.search_markets(term)
    print(f"{term!r}: {len(markets)} markets")
    for m in markets:
        instruments[m["epic"]] = m

print(f"\n{len(instruments)} unique instruments across {len(SEARCH_TERMS)} search terms")


Authenticated OK
'cash': 50 markets
'USD': 34 markets
'EUR': 42 markets
'GBP': 50 markets
'gold': 50 markets
'silver': 19 markets
'oil': 50 markets
'natural gas': 50 markets
'copper': 23 markets
'bitcoin': 50 markets

404 unique instruments across 10 search terms


## 4. View

Straight from the API response - whatever fields IG returned for each market
(typically `epic`, `instrumentName`, `expiry`, plus a live snapshot: `bid`,
`offer`, `marketStatus`, ...).

In [7]:
try:
    import pandas as pd
except ModuleNotFoundError:
    print("pandas not installed - run:  %pip install pandas")
    df = None
else:
    if not instruments:
        print("No instruments found - check SEARCH_TERMS and the account/environment.")
        df = pd.DataFrame()
    else:
        df = pd.DataFrame(instruments.values()).sort_values("instrumentName").reset_index(drop=True)
    display(df)


,epic,instrumentName,instrumentType,expiry,high,low,percentageChange,netChange,updateTime,updateTimeUTC,bid,offer,delayTime,streamingPricesAvailable,marketStatus,scalingFactor
0,UA.D.ATPG.CASH.IP,ATP Oil & Gas Corp,SHARES,-,NaN,NaN,0.00,0.00000,21:00:00,11:00:00,NaN,NaN,0,False,CLOSED,100
1,CS.D.AUDEUR.CFD.IP,AUD/EUR,CURRENCIES,-,0.62100,0.61866,0.21,0.00129,13:34:55,03:34:55,0.62080,0.62095,0,True,TRADEABLE,10000
2,CS.D.AUDGBP.CFD.IP,AUD/GBP,CURRENCIES,-,0.53361,0.53137,0.23,0.00120,13:34:55,03:34:55,0.53337,0.53352,0,True,TRADEABLE,10000
3,CS.D.AUDGBP.MINI.IP,AUD/GBP Mini,CURRENCIES,-,0.53361,0.53137,0.23,0.00120,13:34:55,03:34:55,0.53337,0.53352,0,True,TRADEABLE,10000
4,CS.D.AUDUSD.CFD.IP,AUD/USD,CURRENCIES,-,0.71316,0.71055,0.31,0.00217,13:34:56,03:34:56,0.71305,0.71311,0,True,TRADEABLE,10000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
399,IX.D.SUNCL.UMA.IP,Weekend Oil - US Crude (A$1),COMMODITIES,-,9802.00000,9634.40000,-0.93,-91.40000,13:34:59,03:34:59,9659.70000,9709.70000,0,True,EDITS_ONLY,1
400,IX.D.SUNGOLD.IFA.IP,Weekend Spot Gold (A$1),CURRENCIES,-,4368.10000,4332.10000,-0.03,-1.20000,13:34:58,03:34:58,4348.00000,4353.00000,0,True,EDITS_ONLY,1
401,AB.D.ERWAU.CASH.IP,West Coast Silver Limited,SHARES,-,0.08900,0.08600,4.76,0.00400,13:31:33,03:31:33,NaN,NaN,0,False,TRADEABLE,100
402,KA.D.EGASLN.CASH.IP,WisdomTree European Natural Gas,SHARES,-,NaN,NaN,0.00,0.00000,01:29:58,15:29:58,NaN,NaN,0,False,EDITS_ONLY,1


## 5. Save

Every run writes a new timestamped CSV; nothing is cleaned up
automatically.

In [8]:
if not SAVE_CSV:
    print("SAVE_CSV is False - skipping CSV write")
elif df is None or df.empty:
    print("Nothing to save.")
else:
    from datetime import datetime

    csv_path = cfg.data_dir / f"ig_instruments_{datetime.now():%Y%m%d%H%M%S}.csv"
    df.to_csv(csv_path, index=False)
    print(f"wrote {csv_path.resolve()}  ({len(df)} rows)")


wrote C:\repos\github.com\jupyter-notebooks\data\ig_instruments_20260918133508.csv  (404 rows)
